In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
from collections import Counter
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('dataset/IMDB Dataset.csv')
print(df.info())
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['sentiment'] = df['sentiment'].replace(["positive", "negative"], [1,0]).astype("int64")
df['sentiment']

In [ ]:
x_data = df['review']
y_data = df['sentiment']
print(f'영화 리뷰의 개수: {len(x_data)}')
print(f'레이블의 개수: {len(y_data)}')

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, stratify=y_data)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train)

print(f'훈련용 리뷰의 개수: {len(x_train)}')
print(f'검증용 리뷰의 개수: {len(x_val)}')
print(f'테스트용 리뷰의 개수: {len(x_test)}')
print('----------------------------------')
print(f'훈련용 레이블의 긍정 비율: {y_train.sum()/len(y_train):.2f}')
print(f'검증용 레이블의 긍정 비율: {y_val.sum()/len(y_val):.2f}')
print(f'테스트용 레이블의 긍정 비율: {y_test.sum()/len(y_test):.2f}')

In [ ]:
def tokenize(sentences):
    tokenized_sentences = []
    for sentence in tqdm(sentences):
        tokenized_sentence = word_tokenize(sentence)
        tokenized_sentence = [word.lower() for word in tokenized_sentence]
        tokenized_sentences.append(tokenized_sentence)
    return tokenized_sentences

In [ ]:
tokenized_x_train = tokenize(x_train)
tokenized_x_val = tokenize(x_val)
tokenized_x_test = tokenize(x_test)

In [ ]:
[print(sent) for sent in tokenized_x_train[:3]]

In [ ]:
word_list = [word for sentecne in tokenized_x_train for word in sentecne]
word_counts = Counter(word_list)
print(f'단어 집합의 크기: {len(word_counts)}')
print(f"자주 등장하는 단어의 등장 횟수: {word_counts.most_common()[:10]}")

In [ ]:
vocabs = sorted(word_counts, key=word_counts.get, reverse=True)
print(f'단어 집합의 크기: {len(vocabs)}')
print(f'자주 등장하는 단어 10개: {vocabs[:10]}')

In [ ]:
# 등장 횟수가 작은 단어들을 제거
threshold = 3
total_words, total_freq = len(vocabs), 0
rare_words, rare_freq = 0, 0

for word, count in word_counts.items():
    total_freq += count
    if count < threshold:
        rare_words += 1
        rare_freq += count

print(f'단어 집합의 크기: {total_words}')
print(f'등장 빈도가 {threshold}번 이하인 희귀 단어의 개수: {rare_words}')
print(f'희귀 단어 비율: {rare_words/total_words*100:.2f}%')
print(f'전체 등장 빈도에서 희귀 단어가 차지하는 비율: {rare_freq/total_freq*100:.2f}%')

In [ ]:
vocabs = vocabs[:total_words - rare_words]
word_to_index = {"<PAD>": 0, "<OOV>": 1} | {word: index + 2 for index, word in enumerate(vocabs)}
vocab_size = len(word_to_index)
print(f'단어 집합의 크기: {vocab_size}')

In [ ]:
# 텍스트 시퀀스를 정수 시퀀스로 변환
def encode_sequences(tokenized_sentences, word_to_index):
    encoded_sentences = []
    for sentences in tokenized_sentences:
        index_sequence = []
        for word in sentences:
            index_sequence.append(word_to_index.get(word, 1))
        encoded_sentences.append(index_sequence)
        
    return encoded_sentences

def decode_sequences(encoded_sentences, index_to_word):
    decoded_sentences = []
    for sentence in encoded_sentences:
        word_sequence = []
        for code in sentence:
            word_sequence.append(index_to_word.get(code))
        decoded_sentences.append(word_sequence)
    
    return decoded_sentences

index_to_word = {value: key for key, value in word_to_index.items()}

In [ ]:
encoded_x_train = encode_sequences(tokenized_x_train, word_to_index)
encoded_x_val = encode_sequences(tokenized_x_val, word_to_index)
encoded_x_test = encode_sequences(tokenized_x_test, word_to_index)

In [ ]:
[print(sent) for sent in encoded_x_train[:3]]

In [ ]:
decoding_test = decode_sequences(encoded_x_train[:3], index_to_word)
[print(decode) for decode in decoding_test]

In [ ]:
# 인코딩된 시퀀스를 적절한 길이로 패딩
print(f'리뷰의 최대 길이: {max(len(review) for review in encoded_x_train)}')
print(f'리뷰의 평균 길이: {np.mean([len(review) for review in encoded_x_train]): 2f}')
plt.hist([len(review) for review in encoded_x_train], bins=50, log=True)
plt.xlabel('length of samples(log)')
plt.ylabel('number of samples')
plt.show()



In [ ]:
def ratio_below_threshold_len(threshold, data):
    count = sum([1 for entry in data if len(entry) < threshold])
    return f'{count/len(data)*100:0.2f}%'

In [ ]:
max_len = 1000
print(f'문장 길이가 {max_len} 이하인 샘플의 비율: {ratio_below_threshold_len(max_len, encoded_x_train)}')

In [ ]:
def pad_sequences(encoded_sentences, threshold):
    padded_sequences = np.zeros((len(encoded_sentences), threshold), dtype=int)
    for index, sentence in enumerate(encoded_sentences):
        if sentence:
            length = len(sentence)
            padded_sequences[index, :length] = np.array(sentence)[:threshold]
    return padded_sequences

In [ ]:
padded_x_train = pad_sequences(encoded_x_train, max_len)
padded_x_val = pad_sequences(encoded_x_val, max_len)
padded_x_test = pad_sequences(encoded_x_test, max_len)

print(f'훈련 데이터의 크기: {padded_x_train.shape}')
print(f'검증 데이터의 크기: {padded_x_val.shape}')
print(f'테스트 데이터의 크기: {padded_x_test.shape}')

In [ ]:
# 모델 작성
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.xpu.is_available():
    device = "xpu"
    
print(f'사용 기기: {device}')

In [ ]:
train_data_tensor = torch.tensor(padded_x_train).to(torch.int64)
val_data_tensor = torch.tensor(padded_x_val).to(torch.int64)
test_data_tensor = torch.tensor(padded_x_test).to(torch.int64)
train_label_tensor = torch.tensor(np.array(y_train))
val_label_tensor = torch.tensor(np.array(y_val))
test_label_tensor = torch.tensor(np.array(y_test))

print(f'학습 데이터: {train_data_tensor.shape}')
print(f"학습 레이블: {train_label_tensor.shape}")

In [ ]:
class GRU_Classifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, output_dim):
        super(GRU_Classifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, batch):
        embedded = self.embedding(batch)
        gru_out, hidden = self.gru(embedded)
        last_hidden = hidden[-1]
        logits = self.fc(last_hidden)
        return logits

In [ ]:
def load_data(data, label, batch_size=512):
    dataset = torch.utils.data.TensorDataset(data, label)
    dataloader = torch.utils.data.DataLoader(dataset, shuffle=True, batch_size=batch_size)
    return dataloader

In [ ]:
train_dataloader = load_data(train_data_tensor, train_label_tensor, batch_size = 64)
val_dataloader = load_data(val_data_tensor, val_label_tensor)
test_dataloader = load_data(test_data_tensor, test_label_tensor)

print(f'총 훈련 배치의 수: {len(train_dataloader)}')

In [ ]:
embedding_dim = 128
hidden_dim = 256
num_layers = 2
output_dim = 2
learning_rate = 0.001
num_epochs = 15

model = GRU_Classifier(vocab_size, embedding_dim, hidden_dim, num_layers, output_dim)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
def calculate_accuracy(logits, labels):
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    return correct/labels.size(0)

def evaluate(model, valid_dataloader, criterion, device):
    val_loss = 0
    val_correct = 0
    val_size = 0
    
    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in valid_dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            batch_size = batch_y.size(0)
            
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item()
            val_correct += calculate_accuracy(logits, batch_y) * batch_size
            val_size += batch_size
    
    val_accuracy = val_correct / val_size
    val_loss /= len(valid_dataloader)
    
    return val_loss, val_accuracy

In [ ]:
# 학습
best_val_loss = float("inf")

for epoch in range(num_epochs):
    time_start = time.time()
    train_loss = 0
    train_correct = 0
    train_size = 0
    model.train()
    
    for batch_x, batch_y in tqdm(train_dataloader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        batch_size = batch_y.size(0)

        #forward pass
        logits = model(batch_x)
        
        #compute loss
        loss = criterion(logits, batch_y)
        
        #backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # calculateing training metrics
        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_size
        train_size += batch_size
    
    train_accuracy = train_correct / train_size
    train_loss /= len(train_dataloader)
    
    val_loss, val_accuracy = evaluate(model, val_dataloader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}: ({(time.time() - time_start):0.2f}s)')
    print(f'train loss: {train_loss:0.4f}, train_accuracy: {train_accuracy:0.4f}')
    print(f'validation loss: {val_loss:0.4f}, validation_accuracy: {val_accuracy:0.4f}')
    
    if val_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/best_model_checkpoint_diy.pth')

In [ ]:
model.load_state_dict(torch.load('models/best_model_checkpoint_diy.pth'))
model.to(device)

In [ ]:
# 검증 데이터에 대한 정확도와 손실 계산
val_loss, val_accuracy = evaluate(model, val_dataloader, criterion, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

In [ ]:
lable_parse = {0: "부정", 1: "긍정"}

def predict(text, model, word_to_index, lable_parse):
    model.eval()
    tokens = word_tokenize(text)
    token_indices = [word_to_index.get(token.lower(), 1) for token in tokens]
    
    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
    
    predicted = torch.argmax(logits, dim=1)
    return lable_parse[predicted.item()]

In [ ]:
test_input = "This movie was just way too overrated. The fighting was not professional and in slow motion. I was expecting more from a 200 million budget movie. The little sister of T.Challa was just trying too hard to be funny. The story was really dumb as well. Don't watch this movie if you are going because others say its great unless you are a Black Panther fan or Marvels fan."

predict(test_input, model, word_to_index, lable_parse)

In [ ]:
class CNN_Classifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, kernel_size, num_filters):
        super(CNN_Classifier, self).__init__()
        
        self.num_filters = num_filters
        
        self.word_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        self.conv1d_1st = nn.Conv1d(embedding_dim, hidden_dim, kernel_size, stride=1)
        self.conv1d_2nd = nn.Conv1d(hidden_dim, self.num_filters, kernel_size, stride=1)
        self.dropout= nn.Dropout(0.5)
        self.fc1 = nn.Linear(self.num_filters, num_labels, bias=True)
    
    def forward(self, inputs):
        # (배치 크기, 문장 길이, 임베딩 차원)을 (배치 크기, 임베딩 차원, 문장 길이)로 바꿈
        embedded = self.word_embed(inputs).permute(0, 2, 1)
        # (배치 크기, 커널 개수, 컨볼루션 결과)를 (배치 크기, 컨볼루션 결과, 커널 개수)로 바꿈
        conv1d_1st = self.conv1d_1st(embedded)
        conv1d_2nd = self.conv1d_2nd(conv1d_1st).permute(0, 2, 1)
        max_pooled = F.relu(conv1d_2nd.max(1)[0]) # max(1)은 (values, indices)를 반환
        label_pred = self.fc1(self.dropout(max_pooled))
        
        return label_pred

model = CNN_Classifier(vocab_size, embedding_dim, 256, output_dim, 10, 256)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
best_val_loss = float("inf")

for epoch in range(num_epochs):
    time_start = time.time()
    
    train_loss = 0
    train_correct = 0
    train_size = 0
    
    model.train()
    
    for batch_x, batch_y in tqdm(train_dataloader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        batch_size = batch_y.size(0)
        
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_size
        train_size += batch_size
    
    train_accuracy = train_correct / train_size
    train_loss /= len(train_dataloader)
    
    torch.xpu.empty_cache()
    
    val_loss, val_accuracy = evaluate(model, val_dataloader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}: ({(time.time() - time_start):0.2f}s)')
    print(f'train loss: {train_loss:0.4f}, train_accuracy: {train_accuracy:0.4f}')
    print(f'validation loss: {val_loss:0.4f}, validation_accuracy: {val_accuracy:0.4f}')
    
    if val_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/best_model_checkpoint_cnn_diy.pth')

In [ ]:
model.load_state_dict(torch.load('models/best_model_checkpoint_cnn_diy.pth'))
model.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss, val_accuracy = evaluate(model, val_dataloader, criterion, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

In [ ]:
test_input = "This movie was just way too overrated. The fighting was not professional and in slow motion. I was expecting more from a 200 million budget movie. The little sister of T.Challa was just trying too hard to be funny. The story was really dumb as well. Don't watch this movie if you are going because others say its great unless you are a Black Panther fan or Marvels fan."

predict(test_input, model, word_to_index, lable_parse)